# FlowMetric — воспроизводимый сценарий текущего проекта

Ноутбук показывает основной путь данных на зафиксированном 15-секундном клипе:

1. YOLO11s при размере входа 1280 находит людей, а профиль BoT-SORT повышенной полноты формирует локальные траектории.
2. В полностью локальном режиме идентификаторы остаются локальными. В гибридном режиме Astra получает только галереи пронумерованных фрагментов траекторий, назначает роли и предлагает группы для повторной идентификации. Координаты людей Astra не создаёт и не изменяет.
3. Локальный модуль объединяет фрагменты только при достаточной взвешенной уверенности и минимум двух подтверждениях; сомнительное решение получает статус отказа.
4. Конечный автомат выделяет события, а отдельный автомат состояний определяет ожидание и потребление заказа. Неоднозначные эпизоды обслуживания в этом сценарии проверяются локально по позе, предметам и длительности контакта.
5. Результат сравнивается с `benchmark/ground_truth.json`; сохраняются размеченное видео, `tracks.json`, `events.json` и `benchmark_report.json`.

Сценарий не выбирает первый случайный файл из `data/`: все входы и параметры указаны явно. Полностью локальная ветка не отправляет кадры во внешний API. Гибридная ветка запускается только при наличии API-ключа или кэша Astra и использует отдельный выходной каталог.


## Подготовка окружения

Запускайте Jupyter из корня проекта. Базовая установка:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
pip install jupyter matplotlib
python -m ipykernel install --user --name flowmetric --display-name "FlowMetric"
jupyter notebook notebook.ipynb
```

Выберите ядро **FlowMetric**. Веса Ultralytics могут загрузиться при первом запуске. Для гибридной ветки задайте `VSELLM_API_KEY` или `OPENAI_API_KEY` в `.env`; локальная ветка ключа не требует.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import json
import platform

import cv2
import matplotlib.pyplot as plt

from flowmetric.config import IdentityConfig, PipelineConfig, ServiceConfig, TrackingConfig
from flowmetric.infrastructure.astra import has_api_key
from flowmetric.pipeline import run_pipeline

ROOT = Path.cwd().resolve()
if not (ROOT / "flowmetric").is_dir():
    raise RuntimeError("Запустите notebook.ipynb из корня репозитория FlowMetric")

VIDEO = ROOT / "data" / "hf_20260519_173215_0e502bf6-374c-4039-ab2e-fded1fdf5558.mp4"
GROUND_TRUTH = ROOT / "benchmark" / "ground_truth.json"
CAMERA = ROOT / "config" / "camera.json"
TABLES = ROOT / "config" / "tables.json"
TRACKER = ROOT / "config" / "botsort_recall.yaml"
SERVICE_MODEL = ROOT / "config" / "service_mlp_bootstrap.json"
RUN_DIR = ROOT / "data" / "output" / "notebook"
RUN_DIR.mkdir(parents=True, exist_ok=True)
ASTRA_REID_CACHE = RUN_DIR / "astra_gallery_reid.json"

required = [VIDEO, GROUND_TRUTH, CAMERA, TABLES, TRACKER, SERVICE_MODEL]
missing = [str(path.relative_to(ROOT)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Не найдены обязательные файлы: " + ", ".join(missing))

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "не установлено"

print("Python:", platform.python_version())
print("OpenCV:", cv2.__version__)
print("Ultralytics:", package_version("ultralytics"))
print("PyTorch:", package_version("torch"))
print("Видео:", VIDEO.relative_to(ROOT))
print("Каталог результатов:", RUN_DIR.relative_to(ROOT))
print("API-ключ Astra:", "найден" if has_api_key() else "не найден")
print("Кэш Astra ReID:", "найден" if ASTRA_REID_CACHE.is_file() else "не найден")


## 1. Полностью локальная контрольная ветка

Эта ветка использует тот же детектор, трекер, карту сцены и конечные автоматы, но отключает Astra ReID. Неоднозначные кандидаты обслуживания проверяются локальным модулем: модель позы и детектор предметов извлекают признаки, после чего JSON-MLP принимает бинарное решение. Начальный файл `service_mlp_bootstrap.json` предназначен для проверки работоспособности; для промышленного применения нужен классификатор, обученный на независимой разметке.

Обработка выполняется на каждом кадре и может занять несколько минут. Результаты записываются в `data/output/notebook/local/`.


In [ ]:
tracking = TrackingConfig(
    model_name="yolo11s.pt",
    confidence=0.1,
    iou=0.5,
    image_size=1280,
    tracker=str(TRACKER),
    device="auto",
)
local_service = ServiceConfig(
    verifier="local",
    pose_model="yolo11n-pose.pt",
    object_model="yolo11n.pt",
    classifier=SERVICE_MODEL,
    sample_fps=4.0,
    image_size=640,
)
local_config = PipelineConfig(
    source=VIDEO,
    destination=RUN_DIR / "local" / "tracked.mp4",
    camera=CAMERA,
    tables=TABLES,
    ground_truth=GROUND_TRUTH,
    tracking=tracking,
    identity=IdentityConfig(enabled=False),
    service=local_service,
)

runs = {}
runs["local"] = run_pipeline(local_config)


## 2. Гибридная ветка: галерея Astra + локальная проверка обслуживания

В этой ветке Astra получает контактные листы с пронумерованными фрагментами локальных траекторий. Ответ содержит только группы существующих идентификаторов, роли, уверенность и явные отказы от решения. Локальное объединение требует не менее двух подтверждений и взвешенную уверенность не ниже `0.55`; одновременно видимые пространственно разные люди не объединяются.

Кэш `data/output/notebook/astra_gallery_reid.json` привязан к хешу видео, параметрам выборки кадров, модели, запросу и схеме ответа. Отдельный путь не позволяет случайно взять артефакт другого сценария. Само наличие файла не гарантирует совместимость: `run_pipeline` проверит манифест. Первый запуск требует API-ключ; последующие используют зафиксированный ответ из совместимого кэша.

Проверка обслуживания остаётся локальной, поэтому Astra используется только для повторной идентификации и ролей. Для отдельной проверки обслуживания через Astra задайте `ServiceConfig(verifier="astra", astra_cache=...)`; тогда неоднозначные кандидаты будут проверяться по 4–8 кадрам, но их ID и границы менять нельзя.


In [ ]:
RUN_HYBRID = has_api_key() or ASTRA_REID_CACHE.is_file()

if RUN_HYBRID:
    hybrid_config = PipelineConfig(
        source=VIDEO,
        destination=RUN_DIR / "hybrid_local_service" / "tracked.mp4",
        camera=CAMERA,
        tables=TABLES,
        ground_truth=GROUND_TRUTH,
        tracking=tracking,
        identity=IdentityConfig(
            enabled=True,
            effort="low",
            sample_fps=1.0,
            batch_size=16,
            workers=4,
            match_iou=0.3,
            merge_confidence=0.55,
            minimum_support=2,
            cache=ASTRA_REID_CACHE,
        ),
        service=local_service,
    )
    runs["hybrid+service-local"] = run_pipeline(hybrid_config)
else:
    print("Гибридная ветка пропущена: нет API-ключа и кэша Astra.")
    print("Локальный результат уже находится в runs['local'].")


## 3. Сводка метрик

Строгая F1 требует совпадения типа события, времени и участников. Нестрогая F1 проверяет только тип и время. Общая непрерывность и точность ролей учитывают всех 30 эталонных людей, включая пропущенных; условные показатели относятся только к сопоставленному подмножеству. Временной IoU показывает совпадение интервалов ожидания и потребления заказа.


In [ ]:
def print_summary(name, run):
    report = run["benchmark"]
    identity = report["identity"]
    roles = report["roles"]
    strict = report["events_strict"]
    loose = report["events_loose"]
    states = report["states"]
    print(f"\n{name} [{run['mode']}] on {run['device']}")
    print(f"  итоговые ID: {identity['pred_ids']}")
    print(f"  покрытие людей: {roles['mapped']}/{identity['gt_people']} ({roles['coverage']:.1%})")
    print(f"  непрерывность общая / сопоставленная: {identity['continuity_mean']:.3f} / {identity['continuity_mapped']:.3f}")
    print(f"  роли общие / сопоставленные: {roles['accuracy']:.3f} / {roles['accuracy_mapped']:.3f}")
    print(f"  строгая точность / полнота / F1: {strict['precision']:.3f} / {strict['recall']:.3f} / {strict['f1']:.3f}")
    print(f"  нестрогая F1: {loose['f1']:.3f}")
    print(f"  tIoU ожидания / потребления: {states['waiting']['tiou']:.3f} / {states['consuming']['tiou']:.3f}")

for run_name, run_result in runs.items():
    print_summary(run_name, run_result)


Для текущего эталонного гибридного прогона ориентир из README: покрытие `30/30`, 55 итоговых ID, общая непрерывность `0.855`, точность ролей `0.933`, строгая F1 `0.667`, нестрогая F1 `0.889`, tIoU ожидания `0.748` и потребления `0.746`. Это контрольные значения одного клипа, а не универсальная гарантия: версии библиотек, весов и ответы внешней модели могут влиять на результат.


## 4. Аудит решений

`tracks.json → identity_sync` объясняет объединения, отклонения и отказы повторной идентификации. `events.json → service_candidates` хранит признаки и решения проверки обслуживания, а `customer_state_audit` — основания состояний ожидания и потребления. Следующая ячейка показывает размеры этих разделов, не подменяя их сокращённой интерпретацией.


In [ ]:
for run_name, run_result in runs.items():
    output_dir = Path(run_result["output"]).parent
    tracks_payload = json.loads((output_dir / "tracks.json").read_text(encoding="utf-8"))
    events_payload = json.loads((output_dir / "events.json").read_text(encoding="utf-8"))
    print(f"\n{run_name}: {output_dir.relative_to(ROOT)}")
    print("  траекторий:", len(tracks_payload["tracks"]))
    print("  решений синхронизации:", len(tracks_payload.get("identity_sync", {}).get("identity_decisions", [])))
    print("  событий:", len(events_payload["events"]))
    print("  интервалов обслуживания:", len(events_payload.get("service_intervals", [])))
    print("  кандидатов обслуживания:", len(events_payload.get("service_candidates", [])))
    print("  записей аудита состояний:", len(events_payload.get("customer_state_audit", [])))


## 5. Визуальная проверка

Визуализация открывает кадр из гибридного результата, если он был рассчитан, иначе — из локального. Для проверки другого момента измените `TIME_SECONDS`.


In [ ]:
TIME_SECONDS = 11.6
selected_name = "hybrid+service-local" if "hybrid+service-local" in runs else "local"
selected = runs[selected_name]
video_path = Path(selected["output"])

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f"Не удалось открыть результат: {video_path}")
frame_index = min(int(TIME_SECONDS * selected["fps"]), selected["frames"] - 1)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"Не удалось прочитать кадр {frame_index} из {video_path}")

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(
    f"{selected_name}: кадр {frame_index}, "
    f"локальных траекторий {selected['local_tracks']} → итоговых ID {selected['final_ids']}"
)
plt.axis("off")
plt.show()


## Эквивалентные команды

Те же основные режимы доступны без Jupyter:

```bash
# Полностью локальная обработка с локальной проверкой обслуживания
python track.py --source data/hf_20260519_173215_0e502bf6-374c-4039-ab2e-fded1fdf5558.mp4 \
  --out data/output/notebook_cli/local/tracked.mp4 --no-astra --device auto \
  --model yolo11s.pt --imgsz 1280 --conf 0.1 --iou 0.5 \
  --tracker config/botsort_recall.yaml --camera config/camera.json \
  --tables config/tables.json --service-verifier local \
  --gt benchmark/ground_truth.json

# Гибридная повторная идентификация + локальная проверка обслуживания
python track.py --source data/hf_20260519_173215_0e502bf6-374c-4039-ab2e-fded1fdf5558.mp4 \
  --out data/output/notebook_cli/hybrid/tracked.mp4 --device auto \
  --model yolo11s.pt --imgsz 1280 --conf 0.1 --iou 0.5 \
  --tracker config/botsort_recall.yaml --camera config/camera.json \
  --astra-cache data/output/notebook_cli/astra_gallery_reid.json \
  --tables config/tables.json --service-verifier local \
  --gt benchmark/ground_truth.json
```

Подробные определения метрик и ограничения оценки находятся в `benchmark/README.md`, а полная экспериментальная сетка — в `reports/EXPERIMENTS.md`.
